Required Notice: Copyright (c) 2026 Rickey1990, original project author, rickeyernest1990@gmail.com.

Original project software: [PolyForm Noncommercial License 1.0.0](https://polyformproject.org/licenses/noncommercial/1.0.0); see LICENSE and NOTICE.md. FineWeb-Edu and dependencies retain separate third-party terms.

# PHSMLP - Pooled Head MLP

## FineWeb-Edu 500M · four-way Kaggle study

**Purpose:** assess whether a pooled coordination signal between four independent attention heads and a feed-forward network helps, independently of whether that feed-forward network's parameters are shared across three sequential processing passes.

This is a **2×2 controlled experiment**:

| Variant | Four attention heads | Pooling summary | Feed-forward networks |
|---|---|---|---|
| `standard` | yes | no | 3 separate MLPs |
| `standard_pool` | yes | **yes** | 3 separate MLPs |
| `shared` | yes | no | **one MLP reused 3 times** |
| `shared_pool` | yes | **yes** | **one MLP reused 3 times** |

The pooled head is an **additional learned, differentiable summary path** with a direct bypass. It does **not** collapse or remove the four attention heads. All attention heads operate in parallel on the GPU and pooling is trained end-to-end alongside attention and the MLP.

### What gets trained

- **500,000,000 scored training targets per model**, on the same fixed, tokenizer-measured training stream. Four models means **2 billion training targets in total**.
- Data: [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu), `sample-10BT`, read from a **downloaded Parquet shard attached as a Kaggle input**, not downloaded during training. The notebook extracts 500M tokenizer-measured positions plus separate validation/test samples. A single shard is expected to be substantial, but the code checks it actually contains enough data.
- Model width 64; three processing passes; four heads; 256-neuron MLP; maximum context 512; 4,096-piece training-only SentencePiece BPE vocabulary (actual count is recorded).

### Milestones and scientific controls

- **Every 1M scored targets:** save a resumable `latest.pt` checkpoint (overwrites previous latest), loss/accuracy/diversity/gradient/throughput/VRAM health; stop for non-finite loss or repeated severe validation deterioration.
- **Every 10M:** fact-count, distance, 2/3/4-hop, overwrite, counterfactual, query-swap and copying probes.
- **Every 25M:** expanded probes, pooling ablations, and a retained snapshot.
- **At 500M:** untouched test-set NLL/perplexity, larger independent zero-shot probes, raw predictions, paired bootstrap CIs, 2×2 effect estimates.

**Important:** 2 billion total training targets plus preparing ~1GB of tokenized data is substantial on Kaggle. **Plan for multiple GPU sessions.** The code exits cleanly before its configured session budget and can resume by reattaching the previous notebook output as a Kaggle Input. Do not assume a single session can finish everything.

### One compact ZIP for sharing

The notebook creates **`RESEARCH_RESULTS_UNDER_350MB.zip`** in `/kaggle/working`, normally much smaller than 350MB. It contains code, token/data manifests, all CSVs, raw predictions and summaries. It deliberately excludes the ~1GB training token cache and `.pt` optimizer/checkpoint weights. **That one ZIP is what you should upload back to ChatGPT.**

You can download the report ZIP after any session, even if the full run is unfinished. It is not an analysis of completed training until the model cursors reach 500M.


**Revised v1.1:** fixes the 10M-token 32-fact sampling crash, preflights all probe types before expensive training, records insufficient-vocabulary skips and isolated optional evaluation errors, and keeps the 1M-token checkpoint schedule.


## 1 · Kaggle environment — local-file version

First download [`000_00000.parquet`](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu/resolve/20b2c9f0d743a587e1be1cc836d0ea084fd662d4/sample/10BT/000_00000.parquet) (approximately 2.15 GB) to your computer. In Kaggle create a Dataset from that file, then in your notebook select **Add Input** and attach that Dataset. The file will be mounted somewhere under `/kaggle/input/.../000_00000.parquet`.

Set **Accelerator = GPU**. **You do not need internet access to read the dataset**. Internet may be needed once if the Kaggle Python environment lacks the `datasets`, `sentencepiece` or `pyarrow` packages. If you can already import them, the notebook works without internet.

This version reads the attached Parquet file directly, without Hugging Face streaming. The source file's SHA-256 fingerprint is recorded for reproducibility. The first preparation pass extracts 500M actual SentencePiece scored tokens plus separate validation/test data. If the single shard contains too few tokens, the preparation will stop with an explicit count; then attach another shard and adjust the glob to match both.

In [ ]:
import os, sys, subprocess, platform
from pathlib import Path
print('Python:', platform.python_version())
print('Kaggle working directory:', Path('/kaggle/working').exists())
print('Checking required libraries (local data does not require Hugging Face network access)...')
missing=[]
for package in ('sentencepiece','datasets','pyarrow'):
    try: __import__(package)
    except ImportError: missing.append(package)
if missing:
    print('Missing libraries:',missing,'— attempting pip install; turn Kaggle Internet On if needed')
    subprocess.check_call([sys.executable,'-m','pip','-q','install',*missing])

import torch, numpy as np, sentencepiece, datasets
print('torch:', torch.__version__, '| numpy:', np.__version__, '| datasets:', datasets.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable Kaggle GPU!')
if not torch.cuda.is_available():
    print('WARNING: The 500M-token run will refuse CPU execution. An offline smoke test can run on CPU.')

## 2 · Build the exact, self-contained experiment code

This notebook includes the entire Python implementation. You do not need a separate `.py` upload. The generated file is also copied into the final compact research ZIP, so others can inspect the implementation.

In [ ]:
%%writefile /kaggle/working/research500.py
"""FineWeb-Edu 500M x four architecture factorial experiment.
Embedded in the standalone Kaggle notebook; Python 3.10+, torch, numpy,
SentencePiece, Hugging Face datasets, pandas (optional for plotting).

All targets are scored exactly once per model within a static, frozen training
corpus. Dataset is sourced from locally attached FineWeb-Edu Parquet sample. Neither
validation nor test is used to train the tokenizer or model.
"""
from __future__ import annotations
import os, sys, json, csv, math, time, copy, hashlib, random, signal, shutil, zipfile, traceback, platform, warnings, functools
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import defaultdict
from contextlib import nullcontext
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

VERSION = 'v1.1-500m-factorial-probe-fixes-2026-11-09'
ALL_MODELS = ('standard', 'standard_pool', 'shared', 'shared_pool')
PREFLIGHT_DOCS = 1000
STOP_REQUESTED = False


def request_stop(signum, frame):
    global STOP_REQUESTED
    STOP_REQUESTED = True
    print('STOP REQUESTED: finishing current batch and saving a resumable checkpoint', flush=True)


def atomic_json(path, obj):
    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
    q = p.with_name(p.name + '.tmp')
    q.write_text(json.dumps(obj, indent=2, sort_keys=True, default=str), encoding='utf8')
    q.replace(p)


def read_json(path, default=None):
    p=Path(path)
    if not p.exists(): return default
    return json.loads(p.read_text(encoding='utf8'))


def append_csv(path, record):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    exists=p.exists() and p.stat().st_size>0
    with p.open('a',newline='',encoding='utf8') as f:
        w=csv.DictWriter(f,fieldnames=list(record),extrasaction='ignore')
        if not exists: w.writeheader()
        w.writerow(record)


def sha256_file(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


@dataclass
class Config:
    root: str = '/kaggle/working/fineweb_500m_experiment'
    resume_input_root: str = ''
    dataset_id: str = 'HuggingFaceFW/fineweb-edu'
    dataset_subset: str = 'sample-10BT'
    dataset_revision: str = 'main'  # freeze to resolved commit SHA at collection time
    local_parquet_glob: str = ''  # e.g. /kaggle/input/**/000_00000.parquet; no remote streaming
    scored_tokens: int = 500_000_000
    valid_tokens: int = 2_000_000
    test_tokens: int = 2_000_000
    tokenizer_sample_chars: int = 20_000_000
    vocab: int = 4096
    context: int = 512
    width: int = 64
    hidden: int = 256
    layers: int = 3
    heads: int = 4
    batch: int = 16
    seed: int = 20261109
    max_lr: float = 0.00075
    min_lr: float = 0.00005
    warmup_tokens: int = 4_000_000
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    use_amp: bool = True
    checkpoint_every: int = 1_000_000
    archive_every: int = 25_000_000
    fast_eval_every: int = 1_000_000
    probe_every: int = 10_000_000
    full_eval_every: int = 25_000_000
    models: tuple = ALL_MODELS
    max_session_minutes: int = 600
    max_milestones_per_session: int = 0  # zero = time-limited only
    report_zip_limit_mb: int = 300  # strictly <350 MB including overhead
    auto_stop_on_nan: bool = True
    stop_on_two_severe_val_regressions: bool = True
    smoke: bool = False

    def check(self):
        if self.width % self.heads: raise ValueError('width must divide heads')
        if not 0 < self.vocab <= 65535: raise ValueError('uint16 requires vocab<=65535')
        if self.context < 32: raise ValueError('context too small')
        if self.scored_tokens < 1: raise ValueError('scored_tokens must be >0')
        if self.checkpoint_every < 1: raise ValueError('checkpoint_every must be >0')
        if not self.models or not set(self.models).issubset(ALL_MODELS):
            raise ValueError('Unknown model in '+str(self.models))
        if self.report_zip_limit_mb >= 350: raise ValueError('ZIP limit must be <350 MB')
        return self


def freeze_config(cfg):
    out=Path(cfg.root);out.mkdir(parents=True,exist_ok=True)
    frozen=json.loads(json.dumps(asdict(cfg)))  # normalize tuple/list for JSON resume
    for key in ('max_session_minutes','max_milestones_per_session','resume_input_root'):
        frozen.pop(key,None)
    p=out/'experiment_config.json'
    old=read_json(p)
    if old is None: atomic_json(p,frozen)
    elif old != frozen:
        changes={k:(old.get(k),frozen.get(k)) for k in sorted(set(old)|set(frozen)) if old.get(k)!=frozen.get(k)}
        raise RuntimeError('Experiment configuration differs from previous run. Start a NEW root for a new experiment. Changes: '+str(changes))


def local_parquet_paths(cfg):
    if not cfg.local_parquet_glob:
        return []
    import glob
    paths = [Path(p) for p in sorted(glob.glob(cfg.local_parquet_glob, recursive=True)) if Path(p).is_file()]
    if not paths:
        raise FileNotFoundError('No local Parquet data found for: ' + cfg.local_parquet_glob +
            '. In Kaggle: Add Input > Upload a Dataset containing 000_00000.parquet. ' +
            'Check the actual file path under /kaggle/input and adjust local_parquet_glob.')
    return paths


def source_iterator(cfg, revision=None):
    from datasets import load_dataset
    if cfg.local_parquet_glob:
        # Read directly from the attached Kaggle input; NO requests to Hugging Face.
        paths = local_parquet_paths(cfg)
        ds = load_dataset('parquet', data_files=[str(p) for p in paths], split='train', streaming=True)
    else:
        ds=load_dataset(cfg.dataset_id, name=cfg.dataset_subset, split='train',
                        revision=revision or cfg.dataset_revision, streaming=True)
    for item in ds:
        txt=item.get('text','')
        if isinstance(txt,str) and txt.strip(): yield txt


def resolved_dataset_revision(cfg):
    if cfg.local_parquet_glob:
        paths=local_parquet_paths(cfg)
        # Source fingerprint used for resume integrity; file hashes are saved in the data manifest.
        digests = [{'name':p.name,'size':p.stat().st_size,'sha256':sha256_file(p)} for p in paths]
        data=json.dumps(digests,sort_keys=True).encode('utf8')
        pinned='local-parquet:'+hashlib.sha256(data).hexdigest()
        atomic_json(Path(cfg.root)/'data'/'local_source_manifest.json',{'files':digests,'revision':pinned})
        print('Local Parquet input verified:', ', '.join(p.name for p in paths),flush=True)
        return pinned
    if cfg.dataset_revision != 'main': return cfg.dataset_revision
    try:
        from huggingface_hub import HfApi
        return HfApi().dataset_info(cfg.dataset_id).sha
    except Exception as e:
        print('WARNING: dataset commit could not be pinned:',str(e)[:160],flush=True)
        return 'main'


def prep_tokenizer(cfg, out, revision):
    import sentencepiece as spm
    dir=out/'data';dir.mkdir(parents=True,exist_ok=True)
    path=dir/'spm.model'
    if path.exists():
        sp=spm.SentencePieceProcessor(model_file=str(path))
        if sp.vocab_size() > 65535: raise ValueError('Tokenizer too large for uint16 storage')
        return sp
    sample=dir/'tokenizer_train.txt'
    if not sample.exists():
        print('Collecting tokenizer-training text from the train hash split...',flush=True)
        count=0; docs=0
        partial=sample.with_suffix('.txt.part')
        with partial.open('w',encoding='utf8') as f:
            for text in source_iterator(cfg,revision):
                if split_for_text(text)!='train':continue
                text=text.replace('\0',' ').strip()
                if len(text)>12000:text=text[:12000]
                if not text:continue
                f.write(' '.join(text.split())+'\n')
                count+=len(text);docs+=1
                if count>=cfg.tokenizer_sample_chars:break
        if not count: raise RuntimeError('Tokenizer training sample is empty')
        partial.replace(sample)
        atomic_json(dir/'tokenizer_sample_manifest.json',{'source':cfg.dataset_id,'subset':cfg.dataset_subset,
             'revision':revision,'train_docs':docs,'characters':count})
    print('Training BPE tokenizer...',flush=True)
    spm.SentencePieceTrainer.Train(input=str(sample),model_prefix=str(dir/'spm'),
        vocab_size=cfg.vocab, model_type='bpe',character_coverage=0.9995,
        bos_id=-1,eos_id=1,unk_id=0,pad_id=-1,hard_vocab_limit=False,
        num_threads=4,max_sentence_length=16384)
    sample.unlink(missing_ok=True)
    sp=spm.SentencePieceProcessor(model_file=str(path))
    print('Tokenizer vocab:',sp.vocab_size(),flush=True)
    return sp


def split_for_text(text):
    # Exact duplicate documents assigned to same split; near duplicates remain a caveat.
    h=int.from_bytes(hashlib.sha256(text.encode('utf8','ignore')).digest()[:8],'big') % 1000
    return 'valid' if h<5 else ('test' if h<10 else 'train')


def create_smoke_corpus(cfg):
    """Offline synthetic data ONLY for implementation tests; never scientific results."""
    import sentencepiece as spm
    out=Path(cfg.root); d=out/'data';d.mkdir(parents=True,exist_ok=True)
    txt=d/'mock.txt'
    rng=random.Random(1005)
    words=('Alice Bob Carol David remembered blue green red gold mapping story river '+
           'the bright dog walked home after lunch a small question about science and books').split()
    with txt.open('w') as f:
        for i in range(1800):
            f.write(' '.join(rng.choices(words,k=38))+'\n')
    spm.SentencePieceTrainer.Train(input=str(txt),model_prefix=str(d/'spm'),
        vocab_size=min(cfg.vocab,256),model_type='bpe',bos_id=-1,eos_id=1,
        unk_id=0,pad_id=-1,hard_vocab_limit=False, num_threads=2)
    sp=spm.SentencePieceProcessor(model_file=str(d/'spm.model'))
    vocab=sp.vocab_size();rng=np.random.default_rng(149)
    lengths={'train':cfg.scored_tokens+1,'valid':cfg.valid_tokens,'test':cfg.test_tokens}
    for split,n in lengths.items():
        p=d/f'{split}.bin'
        arr=np.memmap(p,dtype=np.uint16,mode='w+',shape=(n,))
        arr[:]=rng.integers(2,vocab,size=n,dtype=np.uint16);arr.flush();del arr
    manifest={'source':'SYNTHETIC_SMOKE_ONLY','completed':True,'revision':'none','sizes':lengths,
              'sha256_tokenizer':sha256_file(d/'spm.model'),'vocab':vocab,'documents_seen':0,
              'sha256_token_files':{split:sha256_file(d/f'{split}.bin') for split in lengths}}
    atomic_json(d/'manifest.json',manifest)
    txt.unlink(missing_ok=True)
    return manifest


def prepare_corpus(cfg):
    out=Path(cfg.root); out.mkdir(parents=True,exist_ok=True)
    if cfg.smoke:
        if not (out/'data'/'manifest.json').exists():return create_smoke_corpus(cfg)
        return read_json(out/'data'/'manifest.json')
    d=out/'data';d.mkdir(exist_ok=True)
    p=d/'manifest.json'
    manifest=read_json(p)
    expected={'train':cfg.scored_tokens+1,'valid':cfg.valid_tokens,'test':cfg.test_tokens}
    if manifest and manifest.get('completed'):
        if manifest['sizes'] != expected: raise ValueError('Incompatible prepared corpus sizes')
        expected_hashes=manifest.get('sha256_token_files',{})
        for key,n in expected.items():
            f=d/f'{key}.bin'
            if not f.exists() or f.stat().st_size!=2*n:
                raise RuntimeError('Prepared token file missing/wrong size: '+str(f))
            if expected_hashes and sha256_file(f)!=expected_hashes[key]:
                raise RuntimeError('Token corpus fingerprint mismatch: '+str(f))
        print('Using fingerprint-verified prepared token corpus:',p,flush=True)
        return manifest
    revision=(manifest or {}).get('revision') or read_json(d/'source_revision.json',{}).get('revision') or resolved_dataset_revision(cfg)
    atomic_json(d/'source_revision.json', {'dataset_id':cfg.dataset_id,'subset':cfg.dataset_subset,'revision':revision})
    sp=prep_tokenizer(cfg,out,revision)
    if manifest is None:
        manifest={'source':cfg.dataset_id,'subset':cfg.dataset_subset,'revision':revision,
                  'sizes':expected,'vocab':sp.vocab_size(),'sha256_tokenizer':sha256_file(d/'spm.model'),
                  'filled':{'train':0,'valid':0,'test':0},
                  'documents_seen':0,'documents_split':{'train':0,'valid':0,'test':0},
                  'completed':False, 'exact_duplicates_grouped_by_hash':True}
        atomic_json(p,manifest)
    elif manifest['sizes']!=expected or manifest['sha256_tokenizer']!=sha256_file(d/'spm.model'):
        raise RuntimeError('Data manifest/tokenizer changed; refusing unsafe resume')
    mm={}
    for key,n in expected.items():
        f=d/f'{key}.bin'
        if not f.exists():
            with open(f,'wb') as o:o.truncate(n*2)
        if f.stat().st_size!=2*n: raise RuntimeError('Incompatible token cache file '+str(f))
        mm[key]=np.memmap(f,dtype=np.uint16,mode='r+',shape=(n,))
    filled=dict(manifest['filled']);docs_seen=manifest['documents_seen'];docs_split=dict(manifest['documents_split'])
    print('Reading FineWeb-Edu source and building a ~1GB token corpus (local Parquet if configured).',flush=True)
    gen=source_iterator(cfg,revision)
    if docs_seen:
        # Exact deterministic skip from same immutable stream revision.
        print('Restoring at source document',docs_seen,flush=True)
        for _ in range(docs_seen):next(gen)
    since_save=0; timer=time.monotonic()
    try:
        for text in gen:
            docs_seen+=1
            split=split_for_text(text)
            docs_split[split]+=1
            required=expected[split]-filled[split]
            if required>0:
                ids=sp.encode(text,out_type=int)
                if ids:
                    ids.append(sp.eos_id())
                    ids=ids[:required]
                    mm[split][filled[split]:filled[split]+len(ids)]=np.asarray(ids,dtype=np.uint16)
                    filled[split]+=len(ids)
            since_save+=1
            if since_save>=2000 or STOP_REQUESTED:
                for array in mm.values():array.flush()
                manifest.update({'filled':filled,'documents_seen':docs_seen,'documents_split':docs_split})
                atomic_json(p,manifest);since_save=0
                if docs_seen%20000<2000:
                    print('Tokenizer positions:',{k:f'{filled[k]:,}/{expected[k]:,}' for k in expected},flush=True)
                if STOP_REQUESTED:raise KeyboardInterrupt('Stopped during dataset preparation. Resume with same data dir.')
            if all(filled[k]>=expected[k] for k in expected): break
    finally:
        for arr in mm.values():arr.flush()
        manifest.update({'filled':filled,'documents_seen':docs_seen,'documents_split':docs_split})
        atomic_json(p,manifest)
    if not all(filled[k]>=expected[k] for k in expected):
        raise RuntimeError('Stream ended before all split budgets filled. '+str(filled))
    manifest['sha256_token_files']={key:sha256_file(d/f'{key}.bin') for key in expected}
    manifest['completed']=True
    atomic_json(p,manifest)
    print('PREPARED:',{k:f'{v:,}' for k,v in filled.items()},'in seconds',int(time.monotonic()-timer),flush=True)
    return manifest


class MLP(nn.Module):
    def __init__(self,d=64,h=256):
        super().__init__();self.fc1=nn.Linear(d,h);self.fc2=nn.Linear(h,d)
    def forward(self,x):return self.fc2(F.gelu(self.fc1(x)))


class Attention(nn.Module):
    """Four independent heads computed concurrently; optional pooled summary as an EXTRA path."""
    def __init__(self, d=64, heads=4, pooled=False):
        super().__init__(); assert d%heads==0
        self.d=d;self.heads=heads;self.dh=d//heads;self.pooled=pooled
        self.qkv=nn.Linear(d,3*d);self.out=nn.Linear(d,d)
        if pooled:
            self.pool_query=nn.Linear(d,self.dh,bias=False)
            self.pool_out=nn.Linear(self.dh,d)
            self.pool_logit=nn.Parameter(torch.tensor(-1.5))
            nn.init.normal_(self.pool_out.weight,std=0.015)
            nn.init.zeros_(self.pool_out.bias)
        self.pool_mode='normal';self.capture_stats=False;self.last_entropy=None

    def forward(self,x):
        b,t,d=x.shape
        qkv=self.qkv(x).reshape(b,t,3,self.heads,self.dh)
        q,k,v=qkv.unbind(2);q,k,v=[a.transpose(1,2) for a in (q,k,v)]
        heads=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0).transpose(1,2)
        base=self.out(heads.reshape(b,t,d))
        if not self.pooled or self.pool_mode=='off':return base
        query=self.pool_query(x).unsqueeze(2)
        weights=((heads*query).sum(-1)*(self.dh**-0.5)).float().softmax(-1).to(heads.dtype)
        if self.pool_mode=='uniform':weights=torch.full_like(weights,1/self.heads)
        if self.pool_mode=='shifted':weights=weights.roll(shifts=1,dims=-1)
        summary=(heads*weights.unsqueeze(-1)).sum(2)
        contribution=self.pool_logit.sigmoid()*self.pool_out(summary)
        if self.capture_stats:
            probs=weights.detach().float().clamp_min(1e-9)
            self.last_entropy=float((-(probs*probs.log()).sum(-1).mean()/math.log(self.heads)).item())
        return base+contribution


class Block(nn.Module):
    def __init__(self,d,h,heads,pooled,own_ff):
        super().__init__();self.norm_a=nn.LayerNorm(d);self.attn=Attention(d,heads,pooled)
        self.norm_f=nn.LayerNorm(d);self.ff=MLP(d,h) if own_ff else None
    def forward(self,x,shared_ff=None):
        z=x+self.attn(self.norm_a(x));return z+(self.ff if self.ff is not None else shared_ff)(self.norm_f(z))


class Model(nn.Module):
    def __init__(self,cfg,variant,vocab_size=None):
        super().__init__()
        if variant not in ALL_MODELS:raise ValueError(variant)
        self.variant=variant;self.context=cfg.context;self.width=cfg.width
        pooled=variant.endswith('_pool');shared=variant.startswith('shared')
        v=vocab_size or cfg.vocab
        self.embed=nn.Embedding(v,cfg.width);self.pos=nn.Embedding(cfg.context,cfg.width)
        self.shared_ff=MLP(cfg.width,cfg.hidden) if shared else None
        self.blocks=nn.ModuleList([Block(cfg.width,cfg.hidden,cfg.heads,pooled,not shared) for _ in range(cfg.layers)])
        self.final_norm=nn.LayerNorm(cfg.width)
        nn.init.normal_(self.embed.weight,std=0.05)
        nn.init.normal_(self.pos.weight,std=0.01)
    def forward(self,idx):
        b,t=idx.shape
        if t>self.context:raise ValueError('Input longer than context')
        h=self.embed(idx)+self.pos(torch.arange(t,device=idx.device))[None,:,:]
        for block in self.blocks:h=block(h,self.shared_ff)
        return F.linear(self.final_norm(h),self.embed.weight)
    def set_pool_mode(self,mode):
        for block in self.blocks:
            if block.attn.pooled:block.attn.pool_mode=mode
    def pool_stats(self):
        pooled=[b.attn for b in self.blocks if b.attn.pooled]
        if not pooled:return {}
        return {'gate':float(np.mean([float(a.pool_logit.detach().sigmoid()) for a in pooled])),
                'pool_entropy_norm':float(np.mean([a.last_entropy for a in pooled if a.last_entropy is not None]))
                    if any(a.last_entropy is not None for a in pooled) else None}


def param_count(model):return sum(p.numel() for p in model.parameters())


def tensor_from_tokens(arr, offset, b, s, device):
    n=b*s
    x=np.asarray(arr[offset:offset+n],dtype=np.int64).reshape(b,s).copy()
    y=np.asarray(arr[offset+1:offset+n+1],dtype=np.int64).reshape(b,s).copy()
    return torch.as_tensor(x,device=device),torch.as_tensor(y,device=device)


def choose_batch(remain,cfg):
    # Score exactly 'remain' targets, even when a 1M boundary cuts a batch.
    if remain>=cfg.context:
        b=min(cfg.batch,remain//cfg.context)
        return b,cfg.context
    return 1,remain


def learning_rate(seen,cfg):
    if seen<cfg.warmup_tokens:
        return cfg.max_lr*max(0.03,seen/max(cfg.warmup_tokens,1))
    t=min(1.0,(seen-cfg.warmup_tokens)/max(1,cfg.scored_tokens-cfg.warmup_tokens))
    return cfg.min_lr+(cfg.max_lr-cfg.min_lr)*(1+math.cos(math.pi*t))*0.5


def amp_context(cfg,device):
    if device.type!='cuda' or not cfg.use_amp:return nullcontext()
    return torch.autocast(device_type='cuda',dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16))


def make_scaler(cfg,device):
    enabled=device.type=='cuda' and cfg.use_amp and not torch.cuda.is_bf16_supported()
    try:return torch.amp.GradScaler('cuda',enabled=enabled)
    except TypeError:return torch.cuda.amp.GradScaler(enabled=enabled)


def model_grad_norm(model,keyword):
    sums=0.
    for name,p in model.named_parameters():
        if keyword in name and p.grad is not None:sums+=float(p.grad.detach().float().square().sum())
    return sums**0.5


def model_checkpoint_paths(cfg,name):
    d=Path(cfg.root)/'checkpoints'/name
    d.mkdir(parents=True,exist_ok=True)
    return d,d/'latest.pt'


def atomic_torch(path,obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.part')
    torch.save(obj,tmp); tmp.replace(path)


def export_checkpoint(cfg,name,model,optim,scaler,seen,train_steps,manifest_hash,reason='milestone'):
    folder,latest=model_checkpoint_paths(cfg,name)
    state={'version':VERSION,'variant':name,'token_cursor':seen,'train_steps':train_steps,
           'manifest_hash':manifest_hash,'cfg':asdict(cfg),'model':model.state_dict(),
           'optimizer':optim.state_dict(),'scaler':scaler.state_dict(),
           'reason':reason,
           'cpu_rng':torch.get_rng_state(),
           'cuda_rng':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
    atomic_torch(latest,state)
    if seen and (seen%cfg.archive_every==0 or seen==cfg.scored_tokens):
        snap=folder/(f'{seen//1_000_000:04d}M.pt')
        if not snap.exists():shutil.copy2(latest,snap)
    atomic_json(folder/'progress.json',{'variant':name,'token_cursor':seen,'train_steps':train_steps,'reason':reason})


def try_restore(cfg,name,model,optimizer,scaler,manifest_hash):
    folder,latest=model_checkpoint_paths(cfg,name)
    if not latest.exists():return 0,0
    try:state=torch.load(latest,map_location='cpu',weights_only=False)
    except TypeError:state=torch.load(latest,map_location='cpu')
    if state['variant']!=name or state['manifest_hash']!=manifest_hash or state['version']!=VERSION:
        raise RuntimeError('Refusing checkpoint whose dataset/model experiment identity differs')
    model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
    scaler.load_state_dict(state['scaler'])
    # No dropout; per-model stochastic state is not needed in main training path.
    print('RESUMED',name,'at',f"{state['token_cursor']:,}",'tokens',flush=True)
    return int(state['token_cursor']),int(state['train_steps'])


def copy_previous_session(cfg):
    """Optional Kaggle Input source for previous /kaggle/working output version."""
    if not cfg.resume_input_root:return
    src=Path(cfg.resume_input_root)
    if not src.exists():raise FileNotFoundError('resume_input_root does not exist: '+str(src))
    out=Path(cfg.root);out.mkdir(parents=True,exist_ok=True)
    # Locate exact experiment root or a parent with a direct child.
    if not (src/'experiment_config.json').exists():
        found=list(src.glob('**/experiment_config.json'))
        if len(found)!=1:raise RuntimeError('Supply resume_input_root containing one experiment_config.json; found '+str(len(found)))
        src=found[0].parent
    print('Restoring previous Kaggle session from',src,flush=True)
    for folder in ('data','checkpoints','metrics','probes','reports'):
        origin=src/folder
        if not origin.exists():continue
        dest=out/folder;dest.mkdir(exist_ok=True)
        for f in origin.rglob('*'):
            if not f.is_file():continue
            target=dest/f.relative_to(origin)
            if target.exists():continue
            target.parent.mkdir(parents=True,exist_ok=True)
            shutil.copy2(f,target)
    for fn in ('experiment_config.json','experiment_manifest.json','training_log.txt','model_specs.json'):
        f=src/fn
        if f.exists() and not (out/fn).exists():shutil.copy2(f,out/fn)


def sanity_gradients(cfg,device,vocab_size):
    """Checks every head's attention QKV, pooled query, pool projection, gate, and MLP."""
    rows=[]
    for name in cfg.models:
        seed_everything(cfg.seed+17)
        m=Model(cfg,name,vocab_size=vocab_size).to(device)
        x=torch.randint(0,vocab_size,(2,min(cfg.context,32)),device=device)
        with amp_context(cfg,device):
            logits=m(x);loss=F.cross_entropy(logits[:,:-1,:].float().reshape(-1,vocab_size),x[:,1:].reshape(-1))
        loss.backward()
        per_heads=[]
        for block in m.blocks:
            g=block.attn.qkv.weight.grad
            dh=m.width//cfg.heads
            # Q and K gradients are tiny with very small random logits; V and QKV should be nonzero.
            per_heads.append([float(g[h*dh:(h+1)*dh].abs().sum().item()) for h in range(cfg.heads)])
        row={'model':name,'loss':float(loss.item()),
             'attention_nonzero':all(v>0 for layer in per_heads for v in layer),
             'head_gradient_sums':per_heads,
             'ffn_gradient':model_grad_norm(m,'shared_ff' if name.startswith('shared') else '.ff.')}
        if name.endswith('_pool'):
            row.update({'pool_query':model_grad_norm(m,'pool_query'),
                        'pool_out':model_grad_norm(m,'pool_out'),
                        'pool_gate':model_grad_norm(m,'pool_logit')})
            if not all(row[k]>0 for k in ('pool_query','pool_out','pool_gate')):
                raise AssertionError('Pooling gradient missing in '+name+': '+str(row))
        if not row['attention_nonzero'] or row['ffn_gradient']==0:
            raise AssertionError('Attention or MLP gradient missing: '+str(row))
        rows.append(row);del m
    seed_everything(cfg.seed)
    return rows


def evaluate_nll(model,arr,cfg,device,max_positions=8192,offset=0):
    model.eval();total=0.;correct=0;count=0;distinct=set();windows=[]
    length=min(len(arr)-offset-1,max_positions)
    with torch.inference_mode():
        pos=offset
        while pos < offset+length:
            remain=offset+length-pos
            b=min(cfg.batch,max(1,remain//cfg.context))
            s=cfg.context if remain>=cfg.context else remain
            n=min(b*s,remain)
            if n!=b*s:b=1;s=n
            x,y=tensor_from_tokens(arr,pos,b,s,device)
            with amp_context(cfg,device):logits=model(x)
            scores=logits.float()
            loss=F.cross_entropy(scores.reshape(-1,scores.size(-1)),y.reshape(-1),reduction='sum')
            total+=float(loss.item());count+=n
            p=scores.argmax(-1)
            correct+=int((p==y).sum().item())
            distinct.update(p.reshape(-1).unique().detach().cpu().tolist())
            windows.append({'tokens':n,'nll':float(loss.item())/n})
            pos+=n
    return {'nll':total/max(count,1),'ppl':math.exp(min(total/max(count,1),30)),
            'token_accuracy':correct/max(count,1),'unique_predicted_tokens':len(distinct),
            'positions':count,'window_nll':windows}

# ------------------ Held-out synthetic probes ------------------
NAMES='Alice Bob Carol David Emma Frank Grace Henry Iris Jack Kate Liam Mia Noah Olivia Paul Quinn Rose Sam Tara Uma Vera Will Xena Yara Zach Abel Bella Chloe Damon Elsa Felix Gina Hugo Ivan Jane Kira Leo Maya Nina Omar Pia Rex Sara Theo Una Veda Wade Yvonne Zoe Aaron Basil Clara Dina Eric Faye Garth Hope Isaac Juno Kian Lora Mira Nora Opal Peter Rina Seth Tina Vick Wendy Xander'.split()
VALUES='red blue green gold white black brown silver pink orange purple yellow copper violet amber coral indigo pearl ivory teal gray grey cream bronze lime rose violet scarlet olive ruby navy ochre jade iron stone water fire earth wind north south east west alpha beta gamma delta cedar maple oak pine birch moss cloud rain snow sun moon star dawn dusk light dark calm swift brave quiet bright warm cold small large soft sharp clean round true false near far high low first last'.split()


class InsufficientProbeVocabulary(ValueError):
    """An individual synthetic probe is untestable with the current tokenizer.

    This must become a recorded SKIP_VOCAB, not a training-stopping exception.
    """
    def __init__(self, required, available):
        self.required=int(required); self.available=int(available)
        super().__init__(f'Probe needs {required} distinct answer tokens, tokenizer offers {available}')


@functools.lru_cache(maxsize=8)
def eligible_values(sp):
    """All eligible *single-token* answer words, not just the original short list.

    Important fix: the 32-fact probe requires >=32 distinct answers. The old
    implementation expanded the vocabulary only for <8; it crashed at 10M.
    Here we always expand from clean SentencePiece whole-word pieces and verify
    that prepending a space actually encodes as exactly that one token.
    """
    vals=[]; ids=set(); words=set()
    def add(word):
        word=word.strip().lower()
        if len(word)<3 or len(word)>24 or not word.isascii() or not word.isalpha():return
        if word in words or word.capitalize() in NAMES:return
        ids1=sp.encode(' '+word,out_type=int)
        if len(ids1)==1 and ids1[0]!=sp.unk_id() and ids1[0] not in ids:
            vals.append((word,int(ids1[0]))); ids.add(int(ids1[0])); words.add(word)
    for word in VALUES:add(word)
    # Don't wait until fewer than 8 words: 32-fact tests need 32 distinct values.
    for i in range(sp.vocab_size()):
        piece=sp.id_to_piece(i)
        if piece.startswith('▁') and len(piece)>3:
            add(piece[1:])
    return tuple(vals)


@functools.lru_cache(maxsize=8)
def filler_pieces(sp):
    pieces=[]
    for w in (' the',' and',' but',' before',' after',' when',' with',' other',' only',' near',' the'):
        z=sp.encode(w,out_type=int)
        if len(z)==1 and z[0]!=sp.unk_id():pieces.append(int(z[0]))
    if not pieces:
        vals=eligible_values(sp)
        if vals:pieces=[vals[0][1]]
        else:
            # For a pathological tokenizer, select any valid ordinary token.
            for i in range(sp.vocab_size()):
                if i not in {sp.unk_id(),sp.eos_id(),sp.bos_id(),sp.pad_id()}:
                    pieces=[i];break
    return tuple(pieces)


def _compose_probe_core(sp,kind,condition,index,seed,context):
    rng=random.Random(seed*10000019+index*11087+sum(ord(c) for c in kind+str(condition))*983)
    vals=eligible_values(sp)
    names=rng.sample(NAMES,min(len(NAMES),44))
    def pick_values(k):
        if len(vals)<k:raise InsufficientProbeVocabulary(k,len(vals))
        return rng.sample(vals,k)

    def select_options(truth,include=()):
        if len(vals)<4:raise InsufficientProbeVocabulary(4,len(vals))
        picked=[truth]
        for p in include:
            if p[1] not in [x[1] for x in picked] and len(picked)<4:picked.append(p)
        for p in rng.sample(vals,len(vals)):
            if len(picked)>=4:break
            if p[1] not in [x[1] for x in picked]:picked.append(p)
        rng.shuffle(picked)
        return picked
    def qtokens(text):return sp.encode(text,out_type=int)
    fill=filler_pieces(sp)
    def build(facts,question,gap=0):
        gap_tokens=[fill[i%len(fill)] for i in range(gap)]
        return qtokens(facts)+gap_tokens+qtokens(question)
    pid=f'{kind}:{condition}:{index}:{seed}'
    common={'probe_id':pid,'category':kind,'condition':str(condition),'seed':seed,'index':index}
    out=[]
    if kind=='binding':
        n=int(condition)
        chosen=pick_values(n)
        mapping=list(zip(names[:n],chosen))
        rng.shuffle(mapping)
        key,truth=rng.choice(mapping)
        facts=''.join(f'{k} = {v[0]}\n' for k,v in mapping)
        ids=build(facts,f'\nQuestion: What value belongs to {key}?\nAnswer:')
        out=[dict(common,tokens=ids,truth=truth,options=select_options(truth),role='target')]
    elif kind=='distance':
        gap=int(condition)
        val1,val2=pick_values(2)
        key1,key2=names[:2]
        facts=f'{key1} = {val1[0]}\n{key2} = {val2[0]}\n'
        ids=build(facts,f'\nQuestion: What value belongs to {key1}?\nAnswer:',gap)
        out=[dict(common,tokens=ids,truth=val1,options=select_options(val1,[val2]),role='target',gap=gap)]
    elif kind=='multi_hop':
        hops=int(condition)
        val=pick_values(1)[0]
        chain=names[:hops]
        edges=''.join(f'{chain[i]} leads to {chain[i+1]}\n' for i in range(hops-1))
        edges+=f'{chain[-1]} leads to {val[0]}\n'
        extras=rng.sample(names[hops:],min(4, len(names)-hops))
        decoys=pick_values(len(extras))
        rows=[f'{k} leads to {v[0]}\n' for k,v in zip(extras,decoys)]
        rows+=edges.splitlines(keepends=True)
        rng.shuffle(rows)
        ids=build(''.join(rows),f'\nQuestion: Starting from {chain[0]}, what final value is reached?\nAnswer:')
        out=[dict(common,tokens=ids,truth=val,options=select_options(val,decoys),role='target',hops=hops)]
    elif kind=='overwrite':
        vals4=pick_values(4)
        old,new,other,extra=vals4[:4]
        alice,bob,carol=names[:3]
        facts=f'{alice} = {old[0]}\n{bob} = {other[0]}\n{carol} = {extra[0]}\nCorrection: {alice} = {new[0]}\n'
        for role,key,truth in [('updated',alice,new),('unchanged',bob,other)]:
            ids=build(facts,f'\nQuestion: What is the current value of {key}?\nAnswer:')
            out.append(dict(common,probe_id=pid+':'+role,tokens=ids,truth=truth,
                       options=select_options(truth,[old,new,other,extra]),role=role))
    elif kind=='counterfactual':
        old,new,other,extra=pick_values(4)
        alice,bob=names[:2]
        stem=f'{alice} = {old[0]}\n{bob} = {other[0]}\n'
        changed=f'{alice} = {new[0]}\n{bob} = {other[0]}\n'
        opts=select_options(old,[new,other,extra])
        settings=[('target_before',stem,alice,old),('target_after',changed,alice,new),
                  ('unrelated_before',stem,bob,other),('unrelated_after',changed,bob,other)]
        for role,txt,key,truth in settings:
            ids=build(txt,f'\nQuestion: What is the value for {key}?\nAnswer:')
            out.append(dict(common,probe_id=pid+':'+role,pair_id=pid,
                            tokens=ids,truth=truth,options=opts,role=role))
    elif kind=='query_swap':
        val1,val2,extra1,extra2=pick_values(4)
        alice,bob=names[:2]
        facts=f'{alice} = {val1[0]}\n{bob} = {val2[0]}\n'
        opts=select_options(val1,[val2,extra1,extra2])
        for role,key,truth in [('first',alice,val1),('second',bob,val2)]:
            ids=build(facts,f'\nQuestion: What is the value for {key}?\nAnswer:')
            out.append(dict(common,probe_id=pid+':'+role,pair_id=pid,
                     tokens=ids,truth=truth,options=opts,role=role))
    else:raise ValueError(kind)
    for item in out:
        item['valid']=(len(item['tokens'])>=2 and len(item['tokens'])<=context and
                       len({v[1] for v in item['options']})==4)
        item['invalid_reason']='' if item['valid'] else ('SKIP_CONTEXT' if len(item['tokens'])>context else 'SKIP_CANDIDATES')
        item['prompt_hash']=hashlib.sha256(np.asarray(item['tokens'],dtype=np.uint16).tobytes()).hexdigest()[:20]
    return out


def compose_probe(sp,kind,condition,index,seed,context):
    """Non-throwing probe construction: do not waste a 500M training run on a bad probe.

    If a tokenizer cannot support a condition, keep an explicit skipped record
    with its category and expected role; never replace missing answers by
    duplicates (which would corrupt four-choice accuracy).
    """
    try:
        return _compose_probe_core(sp,kind,condition,index,seed,context)
    except InsufficientProbeVocabulary as e:
        pid=f'{kind}:{condition}:{index}:{seed}'
        role_sets={'overwrite':('updated','unchanged'),
                   'counterfactual':('target_before','target_after','unrelated_before','unrelated_after'),
                   'query_swap':('first','second')}
        roles=role_sets.get(kind,('target',))
        return [{'probe_id':pid+(':'+role if len(roles)>1 else ''),
                 'pair_id':pid if len(roles)>1 else '',
                 'category':kind,'condition':str(condition),'seed':seed,
                 'index':index,'role':role,'tokens':[], 'truth':('',-1),
                 'options':[], 'valid':False,'prompt_hash':'',
                 'invalid_reason':'SKIP_VOCAB', 'needed_values':e.required,
                 'available_values':e.available}
                for role in roles]


def probe_specs(stage, cfg, test=False):
    if stage=='mini':
        return [('binding',2,4),('binding',8,4),('overwrite','facts',4),('multi_hop',2,4)]
    if stage=='regular':
        return [('binding',n,8) for n in [2,4,8,16,32]] + \
               [('distance',g,8) for g in [0,16,64,128,256,384]] + \
               [('multi_hop',n,8) for n in [2,3,4]] + \
               [('overwrite','facts',8),('counterfactual','edits',8),('query_swap','swap',8)]
    if stage=='full':
        per=64 if test else 16
        return [('binding',n,per) for n in [2,4,8,16,32]]+ \
               [('distance',g,per) for g in [0,16,64,128,256,384]]+ \
               [('multi_hop',n,per) for n in [2,3,4]]+ \
               [('overwrite','facts',per),('counterfactual','edits',per),('query_swap','swap',per)]
    raise ValueError(stage)


def probe_preflight(sp,cfg):
    """Exercise every probe condition before GPU training; record coverage.

    This deliberately validates, rather than deleting, difficult conditions.
    Conditions without enough clean single-token answers are marked SKIP_VOCAB.
    It also checks exact four-way answer distinctness, paired query structure,
    tokenizer token ID bounds, and context-length handling.
    """
    report={'vocab_size':sp.vocab_size(), 'eligible_answer_values':len(eligible_values(sp)),
            'context':cfg.context,'conditions':[], 'invalid_reasons':{},
            'checked_probes':0,'valid_probes':0,'skipped_probes':0}
    for stage in ('mini','regular','full'):
        for kind,condition,_ in probe_specs(stage,cfg,test=False):
            # Several seeds/indices to catch longer tokenised names/values.
            for i in (0,1,7,31,63):
                probes=compose_probe(sp,kind,condition,i,cfg.seed+700_009,cfg.context)
                if not probes:raise AssertionError('Probe generator returned no records')
                for pr in probes:
                    report['checked_probes']+=1
                    if pr['valid']:
                        report['valid_probes']+=1
                        assert 2 <=len(pr['tokens'])<=cfg.context
                        assert len(pr['options'])==4
                        assert len(set(v[1] for v in pr['options']))==4
                        assert pr['truth'][1] in [v[1] for v in pr['options']]
                        assert all(0<=id_<sp.vocab_size() for id_ in pr['tokens'])
                    else:
                        report['skipped_probes']+=1
                        reason=pr.get('invalid_reason') or 'INVALID'
                        report['invalid_reasons'][reason]=report['invalid_reasons'].get(reason,0)+1
                    report['conditions'].append({'stage':stage,'kind':kind,'condition':str(condition),
                         'index':i,'role':pr['role'],'status':'OK' if pr['valid'] else pr['invalid_reason']})
    atomic_json(Path(cfg.root)/'reports'/'probe_preflight.json',report)
    print('PROBE PREFLIGHT:',report['valid_probes'],'valid,',
          report['skipped_probes'],'skipped; reasons=',report['invalid_reasons'],flush=True)
    return report


def score_one_probe(model,pr,device,cfg):
    # Stable output schema for *both* successful and skipped probes; CSV writers
    # must not silently move columns when the first condition is skipped.
    out={'status':'OK','choice_correct':None,'unrestricted_correct':None,
         'prediction':None,'pred_id':None,'truth':pr['truth'][0] or None,
         'truth_id':pr['truth'][1] if pr['truth'][1]>=0 else None,
         'margin':None,'confidence':None,
         'skip_reason':pr.get('invalid_reason',''),
         'needed_values':pr.get('needed_values'),
         'available_values':pr.get('available_values')}
    if not pr['valid']:
        out['status']=pr.get('invalid_reason') or 'SKIP_CONTEXT'
        return out
    ctx=torch.as_tensor(pr['tokens'],dtype=torch.long,device=device).unsqueeze(0)
    with torch.inference_mode(),amp_context(cfg,device):logits=model(ctx)
    last=logits[0,-1,:].float()
    choices=[v[1] for v in pr['options']]
    scores=last[choices]
    win=int(scores.argmax().item())
    probs=scores.softmax(-1)
    out.update({'choice_correct':int(choices[win]==pr['truth'][1]),
            'unrestricted_correct':int(last.argmax().item()==pr['truth'][1]),
            'prediction':pr['options'][win][0], 'pred_id':choices[win],
            'margin':float((scores[win]-scores.topk(2).values[1]).item()),
            'confidence':float(probs[win].item())})
    return out


def run_probes(model,sp,cfg,device,name,milestone,stage='regular',test=False,pool_mode='normal',max_cases=None):
    model.eval();model.set_pool_mode(pool_mode)
    out=Path(cfg.root)
    folder=out/'probes' / ('test' if test else 'validation')
    folder.mkdir(parents=True,exist_ok=True)
    csvname='probe_rows.csv' if not test else 'final_probe_rows.csv'
    rows=[];skips=defaultdict(int)
    # Deterministic frozen probes, separate validation/test IDs.
    probe_seed=cfg.seed+(99_000_009 if test else 700_009)
    k=0
    for kind,condition,n in probe_specs(stage,cfg,test):
        for j in range(n):
            for p in compose_probe(sp,kind,condition,j,probe_seed,cfg.context):
                if max_cases is not None and k>=max_cases:break
                k+=1
                prediction=score_one_probe(model,p,device,cfg)
                row={'model':name,'token_milestone':milestone,'split':'test' if test else 'validation',
                     'pool_mode':pool_mode,'stage':stage,'probe_id':p['probe_id'],
                     'pair_id':p.get('pair_id',''),'kind':p['category'],
                     'condition':p['condition'],'role':p['role'],'prompt_sha256_20':p['prompt_hash'],
                     'context_tokens':len(p['tokens']), 'candidate_token_ids':';'.join(str(x[1]) for x in p['options']),
                     'correct_token_id':p['truth'][1],**prediction}
                rows.append(row)
                if prediction['status']!='OK':skips[kind]+=1
            if max_cases is not None and k>=max_cases:break
        if max_cases is not None and k>=max_cases:break
    if not rows:return {'count':0,'accuracy':None,'skips':dict(skips)}
    for row in rows:append_csv(folder/csvname,row)
    valid=[r for r in rows if r['choice_correct'] is not None]
    categories={}
    for kind in sorted({r['kind'] for r in rows}):
        rv=[r for r in valid if r['kind']==kind]
        categories[kind]={'n':len(rv),'accuracy':sum(r['choice_correct'] for r in rv)/len(rv) if rv else None,
                          'unrestricted_accuracy':sum(r['unrestricted_correct'] for r in rv)/len(rv) if rv else None}
    summary={'model':name,'tokens':milestone,'stage':stage,'split':'test' if test else 'validation',
         'pool_mode':pool_mode,'count':len(valid),'skipped':len(rows)-len(valid),
         'accuracy':sum(r['choice_correct'] for r in valid)/len(valid) if valid else None,
         'unrestricted_accuracy':sum(r['unrestricted_correct'] for r in valid)/len(valid) if valid else None,
         'categories':categories}
    with (folder/'probe_summaries.jsonl').open('a') as f:f.write(json.dumps(summary)+'\n')
    model.set_pool_mode('normal')
    return summary


def score_suffix(model,prefix,suffix,device,cfg):
    ids=prefix+suffix
    if len(prefix)<1 or len(ids)-1>cfg.context:return None
    x=torch.tensor(ids[:-1],device=device,dtype=torch.long).unsqueeze(0)
    with torch.inference_mode(),amp_context(cfg,device):logits=model(x)
    logp=F.log_softmax(logits[0].float(),dim=-1)
    positions=torch.arange(len(prefix)-1,len(ids)-1,device=device)
    target=torch.tensor(suffix,device=device,dtype=torch.long)
    return -float(logp[positions,target].mean().item())


def run_copying(model,sp,cfg,device,name,milestone,stage='regular',test=False):
    model.eval();rng=random.Random(cfg.seed+(871_999 if test else 67_899))
    values=eligible_values(sp)
    filler=filler_pieces(sp)
    gaps=[0,16,32,64,128,256,384]
    n=4 if stage=='mini' else (8 if stage=='regular' else (64 if test else 16))
    rows=[]
    for gap in gaps:
        for i in range(n):
            if len(values)<4 or not filler:
                rows.append({'model':name,'tokens':milestone,'split':'test' if test else 'validation',
                             'gap':gap,'example_id':i,'nll_true':None,'nll_control':None,
                             'copying_benefit_nats':None,'status':'SKIP_VOCAB'})
                continue
            selected=rng.sample(values,min(8,len(values)))
            true=[t[1] for t in selected[:4]]
            wrong=[t[1] for t in selected[4:8]]
            if len(wrong)!=4:wrong=list(reversed(true))
            start=sp.encode('Earlier words: ',out_type=int)
            end=sp.encode('\nRepeat the earlier words: ',out_type=int)
            mid=[filler[j%len(filler)] for j in range(gap)]
            good=start+true+mid+end
            bad=start+wrong+mid+end
            nll_good=score_suffix(model,good,true,device,cfg)
            nll_bad=score_suffix(model,bad,true,device,cfg)
            rows.append({'model':name,'tokens':milestone,'split':'test' if test else 'validation',
                         'gap':gap,'example_id':i,'nll_true':nll_good,'nll_control':nll_bad,
                         'copying_benefit_nats':nll_bad-nll_good if nll_good is not None and nll_bad is not None else None,
                         'status':'OK' if nll_good is not None and nll_bad is not None else 'SKIP_CONTEXT'})
    out=Path(cfg.root)/'probes'/('test' if test else 'validation')
    for r in rows:append_csv(out/'copying.csv',r)
    usable=[r['copying_benefit_nats'] for r in rows if r['copying_benefit_nats'] is not None]
    return {'copy_mean_gain':float(np.mean(usable)) if usable else None,'copy_n':len(usable),
            'copy_skipped':len(rows)-len(usable)}


def ablations(model,sp,cfg,device,name,milestone,test=False):
    if not name.endswith('_pool'):return None
    arr=np.memmap(Path(cfg.root)/'data'/('test.bin' if test else 'valid.bin'),dtype=np.uint16,mode='r')
    # Same held-out token positions across all modes; also same small paired probe set.
    rows=[]
    for mode in ('normal','uniform','shifted','off'):
        model.set_pool_mode(mode)
        metrics=evaluate_nll(model,arr,cfg,device,max_positions=(8192 if test else 4096))
        result=run_probes(model,sp,cfg,device,name,milestone,stage='mini',test=test,pool_mode=mode)
        row={'model':name,'tokens':milestone,'split':'test' if test else 'validation',
             'mode':mode,'nll':metrics['nll'],'ppl':metrics['ppl'],'mini_probe_accuracy':result['accuracy']}
        rows.append(row)
        append_csv(Path(cfg.root)/'probes'/'pool_ablations.csv',row)
    model.set_pool_mode('normal')
    return rows


def paired_bootstrap(a,b,seed=4,repetitions=1500):
    # model A minus model B, paired binary outcomes by exact probe IDs.
    if not a:return None
    rng=np.random.default_rng(seed)
    diffs=np.array(a,dtype=np.float64)-np.array(b,dtype=np.float64)
    idx=rng.integers(0,len(diffs),size=(repetitions,len(diffs)))
    means=diffs[idx].mean(axis=1)
    return {'n':len(diffs),'mean_difference':float(diffs.mean()),
            'ci_low_95':float(np.quantile(means,0.025)),
            'ci_high_95':float(np.quantile(means,0.975))}


def create_paired_summary(cfg):
    """Paired, per-item comparisons; quantify both pooling and weight sharing."""
    root=Path(cfg.root)
    raw=root/'probes'/'test'/'final_probe_rows.csv'
    if not raw.exists():return None
    by_model=defaultdict(dict)
    with raw.open(newline='',encoding='utf8') as f:
        for row in csv.DictReader(f):
            if row['status']=='OK' and row['pool_mode']=='normal' and row['stage']=='full':
                by_model[row['model']][row['probe_id']]=(int(row['choice_correct']),row['kind'],row['condition'],row['role'])
    comparisons=[('standard_pool','standard','effect_of_pooling_with_private_mlp'),
                 ('shared','standard','effect_of_sharing_without_pooling'),
                 ('shared_pool','shared','effect_of_pooling_with_shared_mlp'),
                 ('shared_pool','standard_pool','effect_of_sharing_with_pooling'),
                 ('shared_pool','standard','combined_architecture_effect')]
    report=[]
    for a,b,description in comparisons:
        if a not in cfg.models or b not in cfg.models:continue
        ra,rb=by_model[a],by_model[b]
        ids=sorted(set(ra)&set(rb))
        if not ids:continue
        for kind in ['ALL']+sorted(set(ra[k][1] for k in ids)):
            subset=[k for k in ids if kind=='ALL' or ra[k][1]==kind]
            result=paired_bootstrap([ra[k][0] for k in subset],[rb[k][0] for k in subset],seed=cfg.seed)
            report.append({'model_a':a,'model_b':b,'contrast':description,'task':kind,**result})
    if all(k in by_model for k in ALL_MODELS):
        common=set.intersection(*(set(by_model[k]) for k in ALL_MODELS))
        ids=sorted(common)
        # Difference in differences: pooled effect with sharing MINUS pooled effect without sharing.
        interactions=[(by_model['shared_pool'][k][0]-by_model['shared'][k][0])-
                      (by_model['standard_pool'][k][0]-by_model['standard'][k][0]) for k in ids]
        z=np.zeros(len(interactions))
        result=paired_bootstrap(interactions,z,seed=cfg.seed+7)
        report.append({'model_a':'shared_pool - shared','model_b':'standard_pool - standard',
                        'contrast':'POOLING_X_SHARING_INTERACTION','task':'ALL',**result})
    atomic_json(root/'reports'/'paired_final_bootstrap.json',report)
    # Matched test-window NLL differences with a window-level (not token-level) bootstrap.
    nll_comparisons=[]
    windows={}
    for name in cfg.models:
        p=root/'reports'/f'{name}_test_windows.json'
        if p.exists():windows[name]=read_json(p)
    for a,b,description in comparisons:
        if a not in windows or b not in windows:continue
        aa=windows[a];bb=windows[b]
        n=min(len(aa),len(bb))
        if n<2:continue
        # Same test windows and scoring positions in each model.
        result=paired_bootstrap([aa[i]['nll'] for i in range(n)],
                                [bb[i]['nll'] for i in range(n)],seed=cfg.seed+11)
        nll_comparisons.append({'model_a':a,'model_b':b,'contrast':description,
                                'unit':'test_window_nll_nats_per_token',**result})
    atomic_json(root/'reports'/'paired_test_nll_bootstrap.json',nll_comparisons)
    return report


# ------------------ Milestone training and evaluation ------------------

def create_models(cfg, vocab, device, manifest_hash):
    models={};opts={};scalers={};seen={};steps={}
    seed_everything(cfg.seed)
    baseline=Model(cfg,'standard',vocab_size=vocab)
    base=baseline.state_dict()
    for name in cfg.models:
        seed_everything(cfg.seed+1000+sum(map(ord,name)))
        m=Model(cfg,name,vocab_size=vocab)
        # Copy all matching weights (embeddings, heads, layer norm, private FFN)
        # to reduce variance from unrelated initial random projections.
        state=m.state_dict()
        for k,v in state.items():
            if k in base and base[k].shape==v.shape:state[k]=base[k].clone()
        m.load_state_dict(state)
        if name.startswith('shared'):
            m.shared_ff.load_state_dict(baseline.blocks[0].ff.state_dict())
        m.to(device)
        opt=torch.optim.AdamW(m.parameters(),lr=cfg.max_lr,betas=(0.9,0.95),
                              eps=1e-8,weight_decay=cfg.weight_decay)
        scaler=make_scaler(cfg,device)
        cursor,nstep=try_restore(cfg,name,m,opt,scaler,manifest_hash)
        models[name]=m;opts[name]=opt;scalers[name]=scaler;seen[name]=cursor;steps[name]=nstep
    del baseline
    atomic_json(Path(cfg.root)/'model_specs.json',{
        name:{'parameters':param_count(m),'parameters_by_module':{
          'embedding_and_positional':sum(p.numel() for n,p in m.named_parameters() if n.startswith(('embed','pos'))),
          'attention_and_pooling':sum(p.numel() for n,p in m.named_parameters() if '.attn.' in n),
          'mlp':sum(p.numel() for n,p in m.named_parameters() if ('.ff.' in n or 'shared_ff' in n))},
          'share_ff':name.startswith('shared'),'pooling':name.endswith('_pool')}
        for name,m in models.items()})
    return models,opts,scalers,seen,steps


def train_one_milestone(cfg,name,model,optimizer,scaler,arr,device,old_seen,old_steps,target):
    model.train();seen=old_seen;steps=old_steps
    train_loss=0.;tokens=0;max_grad=0.;last_grad={};
    if device.type=='cuda':torch.cuda.reset_peak_memory_stats()
    start=time.monotonic();last_lr=None
    while seen<target:
        if STOP_REQUESTED:break
        room=min(target-seen,cfg.batch*cfg.context)
        b,s=choose_batch(room,cfg)
        x,y=tensor_from_tokens(arr,seen,b,s,device)
        lr=learning_rate(seen,cfg)
        for pg in optimizer.param_groups:pg['lr']=lr
        optimizer.zero_grad(set_to_none=True)
        with amp_context(cfg,device):
            logits=model(x)
            loss=F.cross_entropy(logits.float().reshape(-1,logits.size(-1)),y.reshape(-1))
        if not torch.isfinite(loss).item():
            raise FloatingPointError('Non-finite loss: '+str(name)+' at scored token '+str(seen))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        gn=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip,error_if_nonfinite=True)
        gn=float(gn.detach().item())
        if not math.isfinite(gn):raise FloatingPointError('Non-finite gradient norm at '+str(seen))
        last_grad={'attention_grad':model_grad_norm(model,'.attn.qkv.'),
                   'mlp_grad':model_grad_norm(model,'shared_ff' if name.startswith('shared') else '.ff.'),
                   'pool_grad':model_grad_norm(model,'pool_query')+model_grad_norm(model,'pool_out')+
                                 model_grad_norm(model,'pool_logit') if name.endswith('_pool') else None}
        max_grad=max(max_grad,gn)
        scaler.step(optimizer);scaler.update()
        n=b*s;seen+=n;steps+=1;tokens+=n;train_loss+=float(loss.item())*n;last_lr=lr
    if device.type=='cuda':torch.cuda.synchronize()
    elapsed=time.monotonic()-start
    return seen,steps,{'train_nll':train_loss/max(tokens,1),'tokens_this_block':tokens,
          'train_elapsed_sec':elapsed,'throughput_targets_per_sec':tokens/max(elapsed,1e-9),
          'max_grad_norm':max_grad,'lr':last_lr,
          'gpu_peak_allocated_mb':float(torch.cuda.max_memory_allocated()/1e6) if device.type=='cuda' else None,
          **last_grad}


def evaluate_milestone(cfg,name,model,sp,valid,device,seen,train_metrics):
    stage= 'full' if seen%cfg.full_eval_every==0 or seen==cfg.scored_tokens else (
            'regular' if seen%cfg.probe_every==0 else 'none')
    n=32768 if stage=='full' else 4096
    for block in model.blocks:block.attn.capture_stats=True
    val=evaluate_nll(model,valid,cfg,device,max_positions=n)
    for block in model.blocks:block.attn.capture_stats=False
    stats=model.pool_stats()
    row={'model':name,'tokens':seen,'milestone_m':round(seen/1e6,3),
         'timestamp_utc':__import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat(),
         'parameters':param_count(model),'val_nll':val['nll'],'val_ppl':val['ppl'],
         'val_token_acc':val['token_accuracy'],'prediction_diversity':val['unique_predicted_tokens'],
         'eval_positions':val['positions'],'eval_stage':stage,
         'pool_gate':stats.get('gate'),'pool_entropy_norm':stats.get('pool_entropy_norm'),
         **train_metrics, 'status':'OK'}
    if not math.isfinite(val['nll']):row['status']='FAIL_NONFINITE_VALIDATION'
    append_csv(Path(cfg.root)/'metrics'/'health.csv',row)
    if stage!='none':
        try:
            probe_result=run_probes(model,sp,cfg,device,name,seen,stage='full' if stage=='full' else 'regular')
            copy_result=run_copying(model,sp,cfg,device,name,seen,'full' if stage=='full' else 'regular')
            m={'model':name,'tokens':seen,'stage':stage,
               'probe_accuracy':probe_result['accuracy'],
               'probe_unrestricted_accuracy':probe_result['unrestricted_accuracy'],
               'probe_count':probe_result['count'],
               'probe_skipped':probe_result['skipped'],**copy_result}
            append_csv(Path(cfg.root)/'metrics'/'probe_milestones.csv',m)
        except Exception as exc:
            # An optional synthetic probe may fail, but that should not destroy
            # expensive model training. The failure is NOT reported as accuracy.
            append_csv(Path(cfg.root)/'metrics'/'evaluation_errors.csv',
                       {'model':name,'tokens':seen,'stage':stage,'part':'probes/copy',
                        'error':repr(exc),'traceback':traceback.format_exc()[-12000:]})
            print('NONFATAL PROBE FAILURE',name,seen,repr(exc),flush=True)
        finally:
            model.set_pool_mode('normal')
    if stage=='full' and name.endswith('_pool'):
        try:ablations(model,sp,cfg,device,name,seen,test=False)
        except Exception as exc:
            append_csv(Path(cfg.root)/'metrics'/'evaluation_errors.csv',
                       {'model':name,'tokens':seen,'stage':stage,'part':'pool_ablations',
                        'error':repr(exc),'traceback':traceback.format_exc()[-12000:]})
            print('NONFATAL ABLATION FAILURE',name,seen,repr(exc),flush=True)
        finally:model.set_pool_mode('normal')
    return row


def final_test(cfg,models,sp,device):
    root=Path(cfg.root);done=root/'reports'/'final_complete.json'
    if done.exists():return read_json(done)
    if not all((root/'checkpoints'/name/'progress.json').exists() and
               read_json(root/'checkpoints'/name/'progress.json')['token_cursor']>=cfg.scored_tokens
               for name in cfg.models):
        print('Final test is locked until all configured models complete the target.',flush=True)
        return None
    test=np.memmap(root/'data'/'test.bin',dtype=np.uint16,mode='r')
    results={}
    # Untouched test set first opened here; earlier stages only use valid.bin.
    for name in cfg.models:
        model=models[name]
        met=evaluate_nll(model,test,cfg,device,max_positions=min(100_000,len(test)-1))
        results[name]={k:v for k,v in met.items() if k!='window_nll'}
        atomic_json(root/'reports'/f'{name}_test_windows.json',met['window_nll'])
        pr=run_probes(model,sp,cfg,device,name,cfg.scored_tokens,stage='full',test=True)
        cp=run_copying(model,sp,cfg,device,name,cfg.scored_tokens,stage='full',test=True)
        results[name].update({'probe_accuracy':pr['accuracy'],
                              'probe_unrestricted_accuracy':pr['unrestricted_accuracy'],
                              **cp})
        if name.endswith('_pool'):ablations(model,sp,cfg,device,name,cfg.scored_tokens,test=True)
        print('FINAL TEST',name,'ppl=',round(results[name]['ppl'],3),
              'probes=',round(pr['accuracy'],4) if pr['accuracy'] is not None else 'NA',flush=True)
    atomic_json(root/'reports'/'final_results.json',results)
    create_paired_summary(cfg)
    atomic_json(done,{'completed':True,'tokens_per_model':cfg.scored_tokens,
          'models':list(cfg.models),'results_sha256':sha256_file(root/'reports'/'final_results.json')})
    return results


def stop_warnings(cfg,name,current):
    path=Path(cfg.root)/'metrics'/'health.csv'
    if not path.exists():return []
    with path.open(newline='') as f:rows=[r for r in csv.DictReader(f) if r['model']==name]
    if len(rows)<3:return []
    last=rows[-3:]
    recent=[float(r['val_nll']) for r in last]
    reasons=[]
    if recent[1]>recent[0]+0.45 and recent[2]>recent[1]+0.45:
        reasons.append('two_severe_validation_regressions')
    if float(last[-1]['prediction_diversity'])<3:reasons.append('prediction_collapse')
    if reasons:append_csv(Path(cfg.root)/'metrics'/'warnings.csv',
                          {'model':name,'tokens':current,'warning':';'.join(reasons)})
    return reasons


def generate_readme(cfg):
    root=Path(cfg.root);reps=root/'reports';reps.mkdir(parents=True,exist_ok=True)
    desc='''# FineWeb-Edu 500M-token × 4-model research audit

This is the compact audit bundle, NOT the token dataset or model checkpoints.
The dataset is HuggingFaceFW/fineweb-edu, config sample-10BT; see manifest
for resolved revision and tokenizer SHA-256. The experiment scores a frozen
set of 500M training next-token targets exactly once per model and uses
hash-separated validation and untouched final test text.

Four-way factorial design: standard, standard+pool, shared FFN, shared FFN+pool.
Three passes, four heads (computed in parallel), one 64-wide state, 256 FFN
hidden neurons; shared variants reuse the same FFN across three passes.
Pooling is an ADDED residual path, never a replacement for normal multi-head
attention. Pooling queries, projections and gates are trained end-to-end.

Main files:
- experiment_config.json / experiment_manifest.json / model_specs.json
- metrics/health.csv: each 1M-token milestone per model
- metrics/probe_milestones.csv: regular/full probe trajectories
- probes/validation/probe_rows.csv: all prompt-level outcomes
- probes/test/final_probe_rows.csv: held-out final prompt-level outcomes
- probes/pool_ablations.csv: uniform/shifted/off vs learned pooling
- probes/**/copying.csv: repeated-phrase copying vs matched controls
- reports/probe_preflight.json: checks every probe category and skipped conditions
- reports/final_results.json: final held-out language test results
- reports/paired_final_bootstrap.json: paired bootstrap differences
- research500.py: exact implementation and evaluation source

Caveats: synthetic binding/reasoning probes are zero-shot, not fine-tuning.
Four-choice accuracy has 25% nominal chance when probes are valid.
Per-case skipped prompts are tracked with reasons; do not silently discard them.
Token cache and checkpoints deliberately excluded from this ZIP (<350MB).
This is a single-seed pilot unless the full experiment is rerun with other seeds.
Ablation-at-evaluation measures dependence on learned pooling, NOT whether
pooling would help if another architecture were retrained without it.

Source: https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu
Dataset licence noted by provider: ODC-BY. Review its original terms.
'''
    (reps/'READ_ME_FIRST.md').write_text(desc)


def export_report(cfg):
    root=Path(cfg.root);root.mkdir(exist_ok=True,parents=True)
    generate_readme(cfg)
    dst=Path('/kaggle/working/RESEARCH_RESULTS_UNDER_350MB.zip') if Path('/kaggle/working').exists() else root.parent/'RESEARCH_RESULTS_UNDER_350MB.zip'
    include=[]
    for fn in ('experiment_config.json','experiment_manifest.json','model_specs.json','research500.py'):
        p=root/fn
        if p.exists():include.append(p)
    if Path(__file__).exists() and Path(__file__).name=='research500.py' and Path(__file__).resolve()!= (root/'research500.py').resolve():
        # This happens in local smoke tests, not Kaggle. Include source by copying.
        shutil.copy2(Path(__file__),root/'research500.py')
        include.append(root/'research500.py')
    for sub in ('metrics','probes','reports'):
        folder=root/sub
        if folder.exists():include.extend(p for p in folder.rglob('*') if p.is_file())
    for fn in ('data/manifest.json','data/source_revision.json','data/tokenizer_sample_manifest.json','data/spm.model','data/spm.vocab'):
        p=root/fn
        if p.exists():include.append(p)
    # Never include .pt, .bin, random text samples or files outside the allowlist.
    include=sorted(set(include))
    tmp=dst.with_suffix('.zip.part')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for p in include:
            if p.suffix in ('.pt','.pth','.bin','.npy','.parquet'):continue
            z.write(p,arcname=str(p.relative_to(root)))
    size=tmp.stat().st_size
    limit=min(cfg.report_zip_limit_mb,349)*1024*1024
    if size>=limit:
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f'Report ZIP {size/1e6:.1f}MB would exceed safe {cfg.report_zip_limit_mb}MB; reduce probe counts')
    tmp.replace(dst)
    print('DOWNLOADABLE REPORT:',dst,'size',round(size/1e6,2),'MB,',len(include),'candidate files',flush=True)
    return dst


def run(cfg):
    cfg.check()
    out=Path(cfg.root);out.mkdir(parents=True,exist_ok=True)
    if cfg.resume_input_root:copy_previous_session(cfg)
    freeze_config(cfg)
    session_start=time.monotonic()
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if not cfg.smoke and device.type!='cuda':
        raise RuntimeError('500M-token Kaggle run requires GPU. Enable Settings -> Accelerator -> GPU.')
    signal.signal(signal.SIGTERM,request_stop)
    if hasattr(signal,'SIGINT'):signal.signal(signal.SIGINT,request_stop)
    print('EXPERIMENT:',VERSION,'device',device,'model variants',cfg.models,flush=True)
    print('Research report ZIP will contain logs/results, NOT 1GB corpus or optimizer checkpoints.',flush=True)
    models={}
    try:
        manifest=prepare_corpus(cfg)
        if not manifest.get('completed'):raise RuntimeError('Corpus incomplete; resume preparation first')
        import sentencepiece as spm
        sp=spm.SentencePieceProcessor(model_file=str(out/'data'/'spm.model'))
        manifest_hash=sha256_file(out/'data'/'manifest.json')
        probe_preflight(sp,cfg)
        if not (out/'reports'/'gradient_check.json').exists():
            checks=sanity_gradients(cfg,device,sp.vocab_size())
            atomic_json(out/'reports'/'gradient_check.json',checks)
            print('Gradient preflight passed:',[z['model'] for z in checks],flush=True)
        models,opts,scalers,seen,steps=create_models(cfg,sp.vocab_size(),device,manifest_hash)
        valid=np.memmap(out/'data'/'valid.bin',dtype=np.uint16,mode='r')
        train=np.memmap(out/'data'/'train.bin',dtype=np.uint16,mode='r')
        atomic_json(out/'experiment_manifest.json',{
          'version':VERSION,'dataset_manifest_sha256':manifest_hash,
          'scored_targets_per_model':cfg.scored_tokens,
          'models':list(cfg.models),'device':str(device),'torch':torch.__version__,
          'numpy':np.__version__,'python':platform.python_version(),
          'experimental_design':'2x2 private/shared x pooled/unpooled',
          'training_schedule':'same frozen contiguous scored token windows for all models, interleaved by 1M milestones',
          'validation_looked_at_during_training':True,'test_only_at_final':True,
          'pool_bypass':True,'attention_all_heads_parallel':True,
          'token_counts_include_repeated_training_source_documents':'not intentionally repeated, exact dupes may exist in source'})
        milestone_blocks_done=0
        while any(seen[k]<cfg.scored_tokens for k in cfg.models):
            for name in cfg.models:
                if seen[name]>=cfg.scored_tokens:continue
                if STOP_REQUESTED:break
                if cfg.max_session_minutes and time.monotonic()-session_start > cfg.max_session_minutes*60:
                    print('SESSION TIME BUDGET REACHED; saving complete checkpoint states',flush=True)
                    return {'completed':False,'cursors':seen}
                model=models[name]
                target=min(cfg.scored_tokens,(seen[name]//cfg.checkpoint_every+1)*cfg.checkpoint_every)
                try:
                    new_seen,new_steps,train_met=train_one_milestone(cfg,name,model,opts[name],scalers[name],train,device,
                                                                      seen[name],steps[name],target)
                except (FloatingPointError,RuntimeError) as e:
                    print('TRAINING FAILURE',name,str(e),flush=True)
                    append_csv(out/'metrics'/'warnings.csv',{'model':name,'tokens':seen[name],
                               'warning':'TRAINING_EXCEPTION:'+repr(e)})
                    raise
                seen[name]=new_seen;steps[name]=new_steps
                export_checkpoint(cfg,name,model,opts[name],scalers[name],new_seen,new_steps,manifest_hash,
                                  reason='interrupted' if STOP_REQUESTED else 'milestone')
                if new_seen==target:
                    row=evaluate_milestone(cfg,name,model,sp,valid,device,new_seen,train_met)
                    flags=stop_warnings(cfg,name,new_seen)
                    if cfg.stop_on_two_severe_val_regressions and flags:
                        raise RuntimeError('Stopping due to requested validation guardrail: '+str(flags))
                    print('MILESTONE',name,f'{new_seen/1e6:.1f}M',
                          'validation_ppl',round(row['val_ppl'],3),
                          'targets/s',round(train_met['throughput_targets_per_sec']),
                          'pool_gate',round(row['pool_gate'],3) if row['pool_gate'] is not None else '-',
                          'warning',','.join(flags) if flags else '-',flush=True)
                    milestone_blocks_done+=1
                if (new_seen%5_000_000==0 or STOP_REQUESTED):export_report(cfg)
                if cfg.max_milestones_per_session and milestone_blocks_done>=cfg.max_milestones_per_session:
                    print('CONFIGURED MILESTONE SESSION LIMIT; resume for the rest',flush=True)
                    return {'completed':False,'cursors':seen}
                if STOP_REQUESTED:break
            if STOP_REQUESTED:break
        complete=all(seen[k]>=cfg.scored_tokens for k in cfg.models)
        if complete:
            final_test(cfg,models,sp,device)
        return {'completed':complete,'cursors':seen}
    finally:
        try:export_report(cfg)
        except Exception as e:print('WARNING report export failed:',repr(e),flush=True)


if __name__=='__main__':
    from argparse import ArgumentParser
    ap=ArgumentParser()
    ap.add_argument('--smoke',action='store_true')
    ap.add_argument('--root',default=None)
    ap.add_argument('--milestones',type=int,default=4)
    args=ap.parse_args()
    if args.smoke:
        cfg=Config(root=args.root or './fineweb500_offline_smoke',scored_tokens=8192,
             valid_tokens=8193,test_tokens=8193,tokenizer_sample_chars=50000,
             vocab=256,context=64,width=32,hidden=64,layers=2,heads=4,
             batch=2,checkpoint_every=2048,archive_every=4096,probe_every=4096,
             full_eval_every=8192,max_session_minutes=0,
             max_milestones_per_session=args.milestones,smoke=True)
    else:
        cfg=Config(root=args.root or Config.root)
    print('RUN RESULT:',run(cfg))

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'py_compile', '/kaggle/working/research500.py'])
print('Source syntax validated.')

## 3 · Optional small offline functionality test

Before spending Kaggle GPU time, test gradients, partial checkpointing and ZIP export on **synthetic random tokens**. This is explicitly **not** a research result. The smoke-run ZIP is removed immediately afterward so it cannot be mistaken for the actual study output.

In [ ]:
RUN_SMOKE_TEST = True
if RUN_SMOKE_TEST:
    import sys
    sys.path.insert(0,'/kaggle/working')
    from research500 import Config, run
    smoke_cfg=Config(root='/kaggle/working/_implementation_smoke_only',
        scored_tokens=8192, valid_tokens=8193, test_tokens=8193,
        vocab=256, tokenizer_sample_chars=50000, context=64, width=32, hidden=64,
        layers=2,heads=4,batch=2,checkpoint_every=2048,archive_every=4096,
        probe_every=4096,full_eval_every=8192,max_session_minutes=0,
        max_milestones_per_session=4,smoke=True)
    smoke_result=run(smoke_cfg)
    assert all(v==2048 for v in smoke_result['cursors'].values()), smoke_result
    # The old notebook failed when the 32-fact probe demanded more words than
    # the tokenizer supplied. Exercise full/regular/mini and all ablation modes.
    from research500 import probe_preflight, compose_probe, eligible_values, Model, run_probes, run_copying, ablations
    import sentencepiece as spm, torch
    sp=spm.SentencePieceProcessor(model_file=str(Path(smoke_cfg.root)/'data'/'spm.model'))
    audit=probe_preflight(sp,smoke_cfg)
    assert audit['checked_probes'] >= 100
    assert any(k=='binding' and c=='32' for k,c in ((z['kind'],z['condition']) for z in audit['conditions']))
    m=Model(smoke_cfg,'shared_pool',vocab_size=sp.vocab_size())
    for stage in ('mini','regular','full'):
        run_probes(m,sp,smoke_cfg,torch.device('cpu'),'shared_pool',2048,stage=stage,max_cases=48)
    run_copying(m,sp,smoke_cfg,torch.device('cpu'),'shared_pool',2048,stage='mini')
    ablations(m,sp,smoke_cfg,torch.device('cpu'),'shared_pool',2048)
    print('All probe types and ablations passed implementation smoke test.')
    Path('/kaggle/working/RESEARCH_RESULTS_UNDER_350MB.zip').unlink(missing_ok=True)
    print('Smoke test passed; temporary smoke report deleted.')

## 4 · Configure the full 500M experiment

**For the first run** leave `RESUME_INPUT_ROOT = ''`. **For a later Kaggle session**, save the notebook output, reattach it as an input source and put its full experiment directory in `RESUME_INPUT_ROOT`. The script copies prior corpus/checkpoints/reports before continuing. Never change tokenizer, architecture or dataset settings inside the same experiment root. To run independent seeds, create a different root.

If you are starting over, use this corrected notebook **from the first cell**. The old 10M crash came from the research probe generator, not your dataset.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0,'/kaggle/working')
from research500 import Config, run, ALL_MODELS

# First run: leave blank. Resume example:
# RESUME_INPUT_ROOT = '/kaggle/input/my-last-notebook-output/fineweb_500m_experiment'
RESUME_INPUT_ROOT = ''

cfg=Config(
    root='/kaggle/working/fineweb_500m_experiment',
    resume_input_root=RESUME_INPUT_ROOT,
    dataset_id='HuggingFaceFW/fineweb-edu',
    dataset_subset='sample-10BT',
    local_parquet_glob='/kaggle/input/**/000_00000.parquet',  # uploaded FineWeb-Edu shard
    scored_tokens=500_000_000,            # PER MODEL, not across all four
    valid_tokens=2_000_000,
    test_tokens=2_000_000,
    tokenizer_sample_chars=20_000_000,
    vocab=4096,
    width=64,hidden=256,layers=3,heads=4,context=512,batch=16,
    max_lr=0.00075,min_lr=0.00005,warmup_tokens=4_000_000,
    models=('standard','standard_pool','shared','shared_pool'),
    checkpoint_every=1_000_000,         # save latest model+optim+scaler every 1M
    probe_every=10_000_000,            # regular research probes
    full_eval_every=25_000_000,       # expanded probes + ablations
    archive_every=25_000_000,         # retained model snapshots (NOT in ZIP)
    seed=20261109,
    max_session_minutes=600,           # stop cooperatively before Kaggle hard timeout
    max_milestones_per_session=0,      # 0: rely on time limit
    report_zip_limit_mb=300,          # hard cap safely below requested 350 MB
    auto_stop_on_nan=True,
    stop_on_two_severe_val_regressions=True,   # conservatively stop if two huge NLL jumps or collapsed predictions
)
cfg.check()
print('Models:',cfg.models)
print('Scored tokens across all models:', cfg.scored_tokens*len(cfg.models))
print('Report path: /kaggle/working/RESEARCH_RESULTS_UNDER_350MB.zip')
print('If resuming, look for attached saved output roots:')
for p in list(Path('/kaggle/input').glob('**/experiment_config.json'))[:8] if Path('/kaggle/input').exists() else []:
    print('  ',p.parent)

In [ ]:
from research500 import local_parquet_paths
source_files=local_parquet_paths(cfg)
print('Attached FineWeb-Edu Parquet files:')
for f in source_files: print(' ', f, f'{f.stat().st_size/1e9:.2f} GB')
assert source_files, 'Attach the downloaded Parquet as a Kaggle Dataset using Add Input.'


## 5 · Prepare dataset and train all four models

This is the real run. It reads/encodes your attached local FineWeb-Edu Parquet once and creates distinct validation/test samples. It runs the four model variants in round-robin 1M-token blocks and resumes automatically from any matching `latest.pt` files.

After each 5M boundary it refreshes the compact report ZIP; a final export runs on clean exit or an ordinary Python exception. A hard Kaggle runtime kill may bypass that last export, so check for the most recent ZIP and checkpoint. Use saved Kaggle notebook output or attach a dataset to continue.

**Do not interpret results from `_implementation_smoke_only` as real training.**

**Preflight:** after preparing the corpus the notebook now checks 2/4/8/16/32-fact tasks and all other probes. Unavailable conditions are counted as skips rather than stopping training. Validation NLL and numerical failures remain strict safeguards.


In [ ]:
from research500 import run
result = run(cfg)
print('SESSION RESULT:', result)
print('This session may stop before 500M. Reattach outputs and resume until all four reach 500M.')

## 6 · Download ONE compact research ZIP (under 350MB)

It includes evaluation rows, all metrics, summary statistics, design metadata, source code, tokenizer and source-revision fingerprint. It does not contain the 1GB token cache or trained weights. If the ZIP is missing, the experiment did not get as far as its export step: check the output and rerun.

**This is the file to upload to ChatGPT for analysis.**

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
report=Path('/kaggle/working/RESEARCH_RESULTS_UNDER_350MB.zip')
if report.exists():
    size_mb=report.stat().st_size/(1024**2)
    print(f'Report ZIP: {size_mb:.2f} MiB, under 350 MiB? {size_mb<350}')
    assert size_mb < 350
    display(FileLink(str(report)))
else:
    print('No report ZIP yet. Review the training cell log.')

## 7 · Inspect learning curves and progress

These values come from the validation set; they are not final test results. Compare matched checkpoints, not different token counts.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
p=Path(cfg.root)/'metrics'/'health.csv'
if p.exists():
    history=pd.read_csv(p)
    print(history.groupby('model').tail(1)[['model','tokens','val_ppl','val_nll','train_nll','throughput_targets_per_sec']].to_string(index=False))
    plt.figure(figsize=(10,5))
    for name,g in history.groupby('model'):
        g=g.sort_values('tokens')
        plt.plot(g.tokens/1e6,g.val_ppl,label=name)
    plt.xlabel('Scored training tokens (millions) per model');plt.ylabel('Validation perplexity ↓')
    plt.title('FineWeb-Edu validation learning curves');plt.legend();plt.grid(alpha=.2)
    plt.show()
else:
    print('Training results not present yet.')

## 8 · Scientific interpretation checklist

The four-way comparison is essential. Ask separately whether **standard+pool beats standard**, whether **shared+pool beats shared**, and whether **sharing at fixed pooling** affects quality. Primary metric: *held-out test NLL at equal scored tokens*. Secondary metrics: parameter count, training throughput and memory; zero-shot binding, corrections, 2–4 hop and copying by distance. Look at all probe counts and skipped cases, as well as confidence intervals.

**A result is promising only if it persists on held-out data.** Extra pooling weights receiving gradients or a deterioration under evaluation-only ablation shows that the model uses pooling, **not** that pooling is superior to separately trained unpooled models. A single seed cannot establish robustness.

**Kaggle persistence:** `/kaggle/working` is not a permanent cross-session filesystem. Use Kaggle **Save Version** and reattach a previous output as an input (or upload its files as a Kaggle Dataset). Set `RESUME_INPUT_ROOT` before running the next session. The report ZIP alone cannot resume training, because it intentionally excludes weights and the 1GB token corpus.

**Source and data:** [FineWeb-Edu dataset card](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) (provider lists ODC-BY license). Hash-separated document splits are reproducible and exact identical text hashes map to the same split, but near-duplicate leakage is still a potential limitation. The data source is the `sample-10BT` subset; the actual token count is defined by the fitted SentencePiece tokenizer, not the dataset's GPT-2 token estimate.
